# 🎬 Movie Rating Prediction

## CodSoft Data Science Internship — Task 2

---

### 📋 Introduction

This notebook builds a regression model to predict IMDb movie ratings based on various movie attributes like genre, director, actors, duration, votes, and release year.

### 🎯 Objective

**Goal:** Predict continuous IMDb ratings (0-10 scale) using machine learning regression algorithms.

### 📊 Dataset

- **Source:** [Kaggle — IMDb India Movies](https://www.kaggle.com/datasets/adrianmcmahon/imdb-india-movies)
- **Size:** ~15,000+ movies
- **Target:** `Rating` (continuous 0-10)

---

## 1. Import Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import warnings
import re
from pathlib import Path

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Model persistence
import joblib

# Settings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

%matplotlib inline

print('All libraries imported successfully!')

---

## 2. Setup Directories

In [ ]:
# Project paths
PROJECT_ROOT = Path('..').resolve()
DATASET_DIR = PROJECT_ROOT / 'dataset'
MODELS_DIR = PROJECT_ROOT / 'models'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'

# Create directories
for d in [MODELS_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Project Root: {PROJECT_ROOT}')
print(f'Dataset Dir:  {DATASET_DIR}')
print(f'Models Dir:   {MODELS_DIR}')
print(f'Outputs Dir:  {OUTPUTS_DIR}')

---

## 3. Load Dataset

In [ ]:
# Try multiple possible filenames
possible_files = ['IMDb_Movies_India.csv', 'IMDb Movies India.csv', 'imdb_india_movies.csv']

df = None
for filename in possible_files:
    filepath = DATASET_DIR / filename
    if filepath.exists():
        df = pd.read_csv(filepath, encoding='latin-1')
        print(f'Dataset loaded from: {filename}')
        break

if df is None:
    raise FileNotFoundError('Please download the dataset and place it in the dataset/ folder')

print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head(10)

---

## 4. Dataset Exploration

In [ ]:
print(f'Shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')

In [ ]:
df.info()

In [ ]:
# Missing values
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})
missing[missing['Missing Count'] > 0].sort_values('Missing %', ascending=False)

In [ ]:
# Duplicate rows
print(f'Duplicate rows: {df.duplicated().sum()}')

In [ ]:
# Target distribution (Rating)
if 'Rating' in df.columns:
    ratings = pd.to_numeric(df['Rating'], errors='coerce')
    print('Rating Statistics:')
    print(ratings.describe())

---

## 5. Exploratory Data Analysis (EDA)

### 5.1 Missing Values Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis', ax=ax)
ax.set_title('Missing Values Heatmap', fontsize=15, fontweight='bold')
fig.savefig(OUTPUTS_DIR / 'missing_values_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Rating Distribution

In [ ]:
ratings = pd.to_numeric(df['Rating'], errors='coerce').dropna()

fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(ratings, bins=20, kde=True, color='#2563eb', ax=ax)
ax.axvline(ratings.mean(), color='red', linestyle='--', lw=2, label=f'Mean: {ratings.mean():.2f}')
ax.axvline(ratings.median(), color='green', linestyle='-', lw=2, label=f'Median: {ratings.median():.2f}')
ax.set_xlabel('IMDb Rating', fontsize=13)
ax.set_ylabel('Count', fontsize=13)
ax.set_title('Distribution of IMDb Ratings', fontsize=15, fontweight='bold')
ax.legend()
fig.savefig(OUTPUTS_DIR / 'rating_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Genre Distribution

In [ ]:
genres = df['Genre'].dropna().apply(lambda x: str(x).split(',')[0].strip())
genre_counts = genres.value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(genre_counts)))
ax.barh(genre_counts.index[::-1], genre_counts.values[::-1], color=colors)
ax.set_xlabel('Number of Movies', fontsize=13)
ax.set_title('Top 15 Genres by Movie Count', fontsize=15, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)
fig.savefig(OUTPUTS_DIR / 'genre_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.4 Movies by Year

In [ ]:
years = df['Year'].astype(str).str.extract(r'(\d{4})')[0]
years = pd.to_numeric(years, errors='coerce')
years = years[(years >= 1950) & (years <= 2025)].dropna()
year_counts = years.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 6))
ax.fill_between(year_counts.index, year_counts.values, alpha=0.3, color='#2563eb')
ax.plot(year_counts.index, year_counts.values, color='#2563eb', lw=2)
ax.set_xlabel('Year', fontsize=13)
ax.set_ylabel('Number of Movies', fontsize=13)
ax.set_title('Movies Released Per Year', fontsize=15, fontweight='bold')
ax.grid(True, alpha=0.3)
fig.savefig(OUTPUTS_DIR / 'movies_by_year.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.5 Duration Distribution

In [ ]:
durations = df['Duration'].astype(str).str.extract(r'(\d+)')[0]
durations = pd.to_numeric(durations, errors='coerce').dropna()
durations = durations[(durations > 0) & (durations < 400)]

fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(durations, bins=30, kde=True, color='#8b5cf6', ax=ax)
ax.axvline(durations.mean(), color='red', linestyle='--', lw=2, label=f'Mean: {durations.mean():.0f} min')
ax.set_xlabel('Duration (minutes)', fontsize=13)
ax.set_ylabel('Count', fontsize=13)
ax.set_title('Distribution of Movie Duration', fontsize=15, fontweight='bold')
ax.legend()
fig.savefig(OUTPUTS_DIR / 'duration_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.6 Votes Distribution

In [ ]:
votes = df['Votes'].astype(str).str.replace(',', '')
votes = pd.to_numeric(votes, errors='coerce').dropna()
votes = votes[votes > 0]

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(np.log10(votes + 1), bins=30, edgecolor='white', alpha=0.7, color='#f59e0b')
ax.set_xlabel('Log10(Votes)', fontsize=13)
ax.set_ylabel('Count', fontsize=13)
ax.set_title('Distribution of Votes (Log Scale)', fontsize=15, fontweight='bold')
fig.savefig(OUTPUTS_DIR / 'votes_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.7 Top Directors

In [ ]:
directors = df['Director'].dropna()
director_counts = directors.value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.Greens(np.linspace(0.3, 0.9, len(director_counts)))
ax.barh(director_counts.index[::-1], director_counts.values[::-1], color=colors)
ax.set_xlabel('Number of Movies', fontsize=13)
ax.set_title('Top 15 Directors by Movie Count', fontsize=15, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)
fig.savefig(OUTPUTS_DIR / 'top_directors.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.8 Top Actors

In [ ]:
actors = df['Actor 1'].dropna()
actor_counts = actors.value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.Oranges(np.linspace(0.3, 0.9, len(actor_counts)))
ax.barh(actor_counts.index[::-1], actor_counts.values[::-1], color=colors)
ax.set_xlabel('Number of Movies', fontsize=13)
ax.set_title('Top 15 Lead Actors by Movie Count', fontsize=15, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)
fig.savefig(OUTPUTS_DIR / 'top_actors.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 6. Data Preprocessing

In [ ]:
df_clean = df.copy()
print(f'Starting shape: {df_clean.shape}')

### 6.1 Clean Year

In [ ]:
def extract_year(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip().replace('(', '').replace(')', '')
    match = re.search(r'(\d{4})', val_str)
    if match:
        year = int(match.group(1))
        if 1900 <= year <= 2030:
            return year
    return np.nan

df_clean['Year'] = df_clean['Year'].apply(extract_year)
print(f'Year cleaned: {df_clean["Year"].notna().sum()} valid entries')

### 6.2 Clean Duration

In [ ]:
def extract_minutes(val):
    if pd.isna(val):
        return np.nan
    match = re.search(r'(\d+)', str(val))
    if match:
        minutes = int(match.group(1))
        if 1 <= minutes <= 600:
            return minutes
    return np.nan

df_clean['Duration_Minutes'] = df_clean['Duration'].apply(extract_minutes)
print(f'Duration cleaned: {df_clean["Duration_Minutes"].notna().sum()} valid entries')

### 6.3 Clean Votes

In [ ]:
def extract_votes(val):
    if pd.isna(val):
        return np.nan
    val_str = re.sub(r'[^\d]', '', str(val))
    if val_str:
        return int(val_str)
    return np.nan

df_clean['Votes'] = df_clean['Votes'].apply(extract_votes)
print(f'Votes cleaned: {df_clean["Votes"].notna().sum()} valid entries')

### 6.4 Clean Rating

In [ ]:
df_clean['Rating'] = pd.to_numeric(df_clean['Rating'], errors='coerce')
df_clean.loc[(df_clean['Rating'] < 0) | (df_clean['Rating'] > 10), 'Rating'] = np.nan
print(f'Rating valid: {df_clean["Rating"].notna().sum()} entries')

### 6.5 Handle Missing Values

In [ ]:
# Drop rows with missing target
before = len(df_clean)
df_clean = df_clean.dropna(subset=['Rating'])
print(f'Dropped {before - len(df_clean)} rows with missing Rating')

# Impute numeric columns
for col in ['Duration_Minutes', 'Votes', 'Year']:
    if col in df_clean.columns:
        median_val = df_clean[col].median()
        df_clean[col].fillna(median_val, inplace=True)
        print(f'{col}: Imputed with median ({median_val:.1f})')

# Fill categorical with 'Unknown'
for col in ['Genre', 'Director', 'Actor 1', 'Actor 2', 'Actor 3', 'Name']:
    if col in df_clean.columns:
        df_clean[col].fillna('Unknown', inplace=True)

### 6.6 Remove Duplicates

In [ ]:
before = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f'Removed {before - len(df_clean)} duplicates')
print(f'Final shape after preprocessing: {df_clean.shape}')

---

## 7. Feature Engineering

### 7.1 Primary Genre

In [ ]:
df_clean['PrimaryGenre'] = df_clean['Genre'].apply(
    lambda x: str(x).split(',')[0].strip() if pd.notna(x) else 'Unknown'
)
le_genre = LabelEncoder()
df_clean['PrimaryGenre_Encoded'] = le_genre.fit_transform(df_clean['PrimaryGenre'])
print(f'Primary genres: {df_clean["PrimaryGenre"].nunique()}')

### 7.2 Genre Count

In [ ]:
df_clean['GenreCount'] = df_clean['Genre'].apply(
    lambda x: len(str(x).split(',')) if pd.notna(x) and str(x).strip() else 0
)
print(f'Genre count range: {df_clean["GenreCount"].min()}-{df_clean["GenreCount"].max()}')

### 7.3 Director Encoding

In [ ]:
top_directors = df_clean['Director'].value_counts().head(50).index.tolist()
df_clean['Director_Category'] = df_clean['Director'].apply(
    lambda x: x if x in top_directors else 'Other'
)
le_director = LabelEncoder()
df_clean['Director_Encoded'] = le_director.fit_transform(df_clean['Director_Category'])

# Director average rating
director_avg = df_clean.groupby('Director')['Rating'].mean().to_dict()
df_clean['Director_AvgRating'] = df_clean['Director'].map(director_avg)
df_clean['Director_AvgRating'].fillna(df_clean['Rating'].mean(), inplace=True)

### 7.4 Actor Encoding

In [ ]:
top_actors = df_clean['Actor 1'].value_counts().head(100).index.tolist()
df_clean['LeadActor_Category'] = df_clean['Actor 1'].apply(
    lambda x: x if x in top_actors else 'Other'
)
le_actor = LabelEncoder()
df_clean['LeadActor_Encoded'] = le_actor.fit_transform(df_clean['LeadActor_Category'])

# Actor average rating
actor_avg = df_clean.groupby('Actor 1')['Rating'].mean().to_dict()
df_clean['LeadActor_AvgRating'] = df_clean['Actor 1'].map(actor_avg)
df_clean['LeadActor_AvgRating'].fillna(df_clean['Rating'].mean(), inplace=True)

### 7.5 Temporal Features

In [ ]:
df_clean['Decade'] = (df_clean['Year'] // 10 * 10).astype(int)
df_clean['MovieAge'] = (2024 - df_clean['Year']).clip(lower=0)
print(f'Decade range: {df_clean["Decade"].min()}-{df_clean["Decade"].max()}')
print(f'Movie age range: {df_clean["MovieAge"].min()}-{df_clean["MovieAge"].max()}')

### 7.6 Vote Features

In [ ]:
bins = [0, 100, 1000, 10000, 100000, float('inf')]
labels = ['VeryLow', 'Low', 'Medium', 'High', 'VeryHigh']
df_clean['VoteCategory'] = pd.cut(df_clean['Votes'], bins=bins, labels=labels)
le_votes = LabelEncoder()
df_clean['VoteCategory_Encoded'] = le_votes.fit_transform(df_clean['VoteCategory'].astype(str))

df_clean['Votes_Log'] = np.log1p(df_clean['Votes'])
print(f'Votes_Log range: {df_clean["Votes_Log"].min():.2f}-{df_clean["Votes_Log"].max():.2f}')

### 7.7 Duration Features

In [ ]:
bins = [0, 90, 120, 150, float('inf')]
labels = ['Short', 'Standard', 'Long', 'VeryLong']
df_clean['DurationCategory'] = pd.cut(df_clean['Duration_Minutes'], bins=bins, labels=labels)
le_duration = LabelEncoder()
df_clean['DurationCategory_Encoded'] = le_duration.fit_transform(df_clean['DurationCategory'].astype(str))

### 7.8 Drop Unused Columns

In [ ]:
cols_to_drop = ['Name', 'Duration', 'Genre', 'Director', 'Actor 1', 'Actor 2', 'Actor 3',
                'PrimaryGenre', 'Director_Category', 'LeadActor_Category', 'VoteCategory', 'DurationCategory']
df_clean = df_clean.drop(columns=[c for c in cols_to_drop if c in df_clean.columns])

print(f'Final shape: {df_clean.shape}')
print(f'\nFeatures: {list(df_clean.columns)}')

### 7.9 Correlation Heatmap

In [ ]:
numeric_df = df_clean.select_dtypes(include=[np.number])

fig, ax = plt.subplots(figsize=(12, 10))
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap', fontsize=15, fontweight='bold')
fig.savefig(OUTPUTS_DIR / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 8. Train-Test Split & Scaling

In [ ]:
X = df_clean.drop(columns=['Rating'])
y = df_clean['Rating']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set: {X_train.shape}')
print(f'Test set:     {X_test.shape}')

In [ ]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print('Features scaled with StandardScaler.')

---

## 9. Model Training & Comparison

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0, random_state=42),
    'Lasso Regression': Lasso(alpha=0.1, random_state=42, max_iter=10000),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = []

print(f'{"Model":25s} | {"MAE":>8s} | {"RMSE":>8s} | {"R²":>8s} | {"CV R²":>10s}')
print('-' * 70)

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='r2')
    
    results.append({
        'Model': name,
        'MAE': round(mae, 4),
        'MSE': round(mse, 4),
        'RMSE': round(rmse, 4),
        'R² (Test)': round(r2, 4),
        'CV R² Mean': round(cv_scores.mean(), 4),
        'CV R² Std': round(cv_scores.std(), 4),
    })
    
    print(f'{name:25s} | {mae:>8.4f} | {rmse:>8.4f} | {r2:>8.4f} | {cv_scores.mean():>6.4f}±{cv_scores.std():.4f}')

comparison_df = pd.DataFrame(results).sort_values('CV R² Mean', ascending=False).reset_index(drop=True)
comparison_df.to_csv(OUTPUTS_DIR / 'model_comparison.csv', index=False)
print('\n--- Model Comparison Table ---')
comparison_df

---

## 10. Hyperparameter Tuning

In [ ]:
top_models = comparison_df.head(2)['Model'].tolist()
top_models = [m for m in top_models if m != 'Linear Regression']
print(f'Top models for tuning: {top_models}')

In [ ]:
param_grids = {
    'Random Forest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [5, 10, 15, None],
        'min_samples_split': [2, 5],
    },
    'Gradient Boosting': {
        'n_estimators': [100, 200],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 5, 7],
    },
}

best_model = None
best_score = -float('inf')
best_name = ''
best_params = {}

for model_name in top_models:
    if model_name not in param_grids:
        continue
    
    print(f'\nTuning: {model_name}...')
    
    if model_name == 'Random Forest':
        base_model = RandomForestRegressor(random_state=42, n_jobs=-1)
    else:
        base_model = GradientBoostingRegressor(random_state=42)
    
    grid = GridSearchCV(
        base_model,
        param_grids[model_name],
        cv=5, scoring='r2', n_jobs=-1, verbose=0
    )
    grid.fit(X_train_scaled, y_train)
    
    print(f'  Best Parameters: {grid.best_params_}')
    print(f'  Best CV R²:      {grid.best_score_:.4f}')
    
    if grid.best_score_ > best_score:
        best_score = grid.best_score_
        best_model = grid.best_estimator_
        best_name = model_name
        best_params = grid.best_params_

print(f'\n{"="*50}')
print(f'BEST MODEL: {best_name}')
print(f'BEST CV R²: {best_score:.4f}')
print(f'BEST PARAMETERS: {best_params}')
print(f'{"="*50}')

---

## 11. Model Evaluation

In [ ]:
y_pred = best_model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f'Model: {best_name}')
print(f'{"─"*50}')
print(f'MAE:   {mae:.4f}  → Average absolute error')
print(f'MSE:   {mse:.4f}  → Mean squared error')
print(f'RMSE:  {rmse:.4f}  → Root mean squared error')
print(f'R²:    {r2:.4f}  → Variance explained')

### 11.1 Prediction vs Actual

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(y_test, y_pred, alpha=0.5, edgecolors='white', linewidth=0.5, s=50)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
ax.annotate(f'R² = {r2:.4f}', xy=(0.05, 0.95), xycoords='axes fraction',
            fontsize=14, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_xlabel('Actual Rating', fontsize=13)
ax.set_ylabel('Predicted Rating', fontsize=13)
ax.set_title('Prediction vs Actual Rating', fontsize=15, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
fig.savefig(OUTPUTS_DIR / 'prediction_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.2 Residual Analysis

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_pred, residuals, alpha=0.5, edgecolors='white', linewidth=0.5)
axes[0].axhline(y=0, color='red', linestyle='--', lw=2)
axes[0].set_xlabel('Predicted Rating', fontsize=12)
axes[0].set_ylabel('Residuals', fontsize=12)
axes[0].set_title('Residuals vs Predicted', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].hist(residuals, bins=30, edgecolor='white', alpha=0.7, color='#2563eb')
axes[1].axvline(x=0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Residuals', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Residual Distribution', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(OUTPUTS_DIR / 'residual_plot.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.3 Feature Importance

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feat_imp = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': importances
    }).sort_values('Importance', ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(feat_imp)))
    ax.barh(feat_imp['Feature'], feat_imp['Importance'], color=colors, edgecolor='white')
    ax.set_xlabel('Importance', fontsize=13)
    ax.set_title('Feature Importance', fontsize=15, fontweight='bold')
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    fig.savefig(OUTPUTS_DIR / 'feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print('\nTop 5 Features:')
    print(feat_imp.tail(5)[['Feature', 'Importance']].to_string(index=False))

---

## 12. Save Model

In [ ]:
model_path = MODELS_DIR / 'movie_rating_model.pkl'
joblib.dump(best_model, model_path)
print(f'Model saved to: {model_path}')

scaler_path = MODELS_DIR / 'scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f'Scaler saved to: {scaler_path}')

---

## 13. Predictions

In [ ]:
predictions_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': np.round(y_pred, 2),
    'Residual': np.round(y_test.values - y_pred, 2),
    'AbsoluteError': np.round(np.abs(y_test.values - y_pred), 2),
})

predictions_df.to_csv(OUTPUTS_DIR / 'predictions.csv', index=False)
print(f'Predictions saved to: {OUTPUTS_DIR / "predictions.csv"}')
print(f'\nWithin ±0.5: {(predictions_df["AbsoluteError"] <= 0.5).mean():.1%}')
print(f'Within ±1.0: {(predictions_df["AbsoluteError"] <= 1.0).mean():.1%}')
predictions_df.head(10)

---

## 14. Conclusion

### Summary

In this project, we built a regression model to predict IMDb movie ratings:

1. **Data Exploration:** Analyzed ~15,000 Indian movies with 10 features
2. **Preprocessing:** Cleaned Year, Duration, Votes; handled missing values
3. **Feature Engineering:** Created 12+ features (director/actor encoding, temporal features, vote categories)
4. **Model Training:** Compared 6 regression algorithms
5. **Hyperparameter Tuning:** Optimized top models with GridSearchCV
6. **Evaluation:** Achieved good predictive performance

### Key Findings

- **Votes** (log-transformed) is the strongest predictor
- **Director reputation** significantly impacts ratings
- **Actor star power** contributes to rating prediction
- **Duration** has a modest positive correlation
- **Genre** influences ratings differently

### Future Improvements

1. Add more features: budget, box office, release month
2. NLP features from movie descriptions
3. Advanced ensemble methods
4. External data (social media sentiment)

---

*Project completed as part of CodSoft Data Science Internship — Task 2*